<a href="https://colab.research.google.com/github/AntonDozhdikov/Demography_migration/blob/main/CFO_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SFO_pipelineV3 — MATD3-only · 10 регионов СФО

Демографический MARL-эксперимент для 10 регионов Сибирского федерального округа.  
Алгоритм: **MATD3** (Multi-Agent Twin Delayed DDPG).  
Горизонт прогноза: 2026–2050.  

## Section 1 — Импорты и конфигурация

In [1]:
# Установка необходимых пакетов
import subprocess, sys

def pip_install(pkg: str):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["scipy", "scikit-learn", "tqdm", "seaborn", "openpyxl"]:
    pip_install(pkg)

print("✓ Базовые пакеты установлены")

✓ Базовые пакеты установлены


In [2]:
# Импорты
import os
import copy
import random
import warnings
import json
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")  # headless
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.signal import savgol_filter
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

warnings.filterwarnings("ignore")

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["figure.dpi"] = 110

print("✓ Импорты загружены")

✓ Импорты загружены


In [3]:
# Глобальные настройки
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True

print("DEVICE:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

BASEOUT  = Path("sfo_marl_output")
PLOTDIR  = BASEOUT / "plots"
LOGDIR   = BASEOUT / "logs"
MODELDIR = BASEOUT / "models"
REPORTDIR = BASEOUT / "reportdata"

for d in [PLOTDIR, LOGDIR, MODELDIR, REPORTDIR]:
    d.mkdir(parents=True, exist_ok=True)

print("OUTPUT DIR:", BASEOUT.resolve())

DEVICE: cuda
GPU: Tesla T4
OUTPUT DIR: /content/sfo_marl_output


In [4]:
# Конфигурационный словарь
CFG = {
    # Данные
    "regions": [
        "Республика Алтай",
        "Республика Бурятия",
        "Республика Тыва",
        "Республика Хакасия",
        "Алтайский край",
        "Забайкальский край",
        "Красноярский край",
        "Иркутская область",
        "Новосибирская область",
    ],
    "datadir": "data/sfo",
    "sheet": "statespace",
    "histend": 2025,
    "forecastend": 2050,
    "trainend": 2012,
    "valend": 2018,

    # Среда
    "dt": 1,
    "horizon": 25,
    "nscenarios": 30,
    "shockstd": 0.02,

    # MARL (MATD3)
    "nagents": 8,
    "actorlr": 1e-3,
    "criticlr": 1e-3,
    "gamma": 0.95,
    "tau": 0.005,
    "buffersize": 50_000,
    "batchsize": 256,
    "hiddendim": 256,
    "nepisodes": 600,
    "maxsteps": 25,
    "warmupsteps": 500,
    "policydelay": 2,
    "targetnoise": 0.2,
    "noiseclip": 0.5,

    # Весовые коэффициенты в награде
    "rewardw": {
        "popgrowth": 0.20,
        "tfr": 0.25,
        "netmigration": 0.15,
        "natincrease": 0.15,
        "housing": 0.10,
        "healthcare": 0.10,
        "costpenalty": -0.05,
    },

    "seed": SEED,
}

print("CFG готов, регионов:", len(CFG["regions"]))

CFG готов, регионов: 9


In [5]:
# Progress helpers для чистого отображения эксперимента в Jupyter
import time
from contextlib import contextmanager
from tqdm.notebook import tqdm

def format_seconds(sec):
    sec = max(0, int(sec))
    h = sec // 3600
    m = (sec % 3600) // 60
    s = sec % 60
    if h > 0:
        return f"{h:d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"

@contextmanager
def region_experiment_bar(regions, desc="SFO experiment"):
    start = time.time()
    bar = tqdm(
        total=len(regions),
        desc=desc,
        leave=True,
        mininterval=1.0,
        dynamic_ncols=True,
        bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}{postfix}]",
    )
    try:
        yield bar, start
    finally:
        bar.close()

def update_region_bar(bar, start_time, done_count, total_count, region_name, region_elapsed=None):
    elapsed = time.time() - start_time
    avg = elapsed / max(done_count, 1)
    remain = avg * max(total_count - done_count, 0)

    postfix = f"region={region_name}"
    if region_elapsed is not None:
        postfix += f" | last={format_seconds(region_elapsed)}"
    postfix += f" | eta_total={format_seconds(remain)}"

    bar.set_postfix_str(postfix, refresh=False)

In [6]:
# Окно загрузки файлов для Colab
from google.colab import files
from pathlib import Path

upload_dir = Path(CFG["datadir"])
upload_dir.mkdir(parents=True, exist_ok=True)

print(f"Загрузи сюда файлы .xlsx /.csv для регионов: {', '.join(CFG['regions'])}")
print(f"Они будут сохранены в каталог: {upload_dir.resolve()}")

uploaded = files.upload()  # выбери все файлы разом

for fname, data in uploaded.items():
    out_path = upload_dir / fname
    with open(out_path, "wb") as f:
        f.write(data)
    print("Сохранено:", out_path)

Загрузи сюда файлы .xlsx /.csv для регионов: Республика Алтай, Республика Бурятия, Республика Тыва, Республика Хакасия, Алтайский край, Забайкальский край, Красноярский край, Иркутская область, Новосибирская область
Они будут сохранены в каталог: /content/data/sfo


Saving Алтайский_край.xlsx to Алтайский_край.xlsx
Saving Забайкальский_край.xlsx to Забайкальский_край.xlsx
Saving Иркутская_область.xlsx to Иркутская_область.xlsx
Saving Красноярский_край.xlsx to Красноярский_край.xlsx
Saving Новосибирская_область.xlsx to Новосибирская_область.xlsx
Saving Республика_Алтай.xlsx to Республика_Алтай.xlsx
Saving Республика_Бурятия.xlsx to Республика_Бурятия.xlsx
Saving Республика_Тыва.xlsx to Республика_Тыва.xlsx
Saving Республика_Хакасия.xlsx to Республика_Хакасия.xlsx
Сохранено: data/sfo/Алтайский_край.xlsx
Сохранено: data/sfo/Забайкальский_край.xlsx
Сохранено: data/sfo/Иркутская_область.xlsx
Сохранено: data/sfo/Красноярский_край.xlsx
Сохранено: data/sfo/Новосибирская_область.xlsx
Сохранено: data/sfo/Республика_Алтай.xlsx
Сохранено: data/sfo/Республика_Бурятия.xlsx
Сохранено: data/sfo/Республика_Тыва.xlsx
Сохранено: data/sfo/Республика_Хакасия.xlsx


## Section 2 — Загрузка и предобработка данных

Для каждого из 10 регионов СФО выполняется:
- загрузка исторического ряда из Excel/CSV (если есть),

- масштабирование признаков (StandardScaler по историческому отрезку).

In [7]:
# Базовые демографические параметры регионов СФО (поправки к синтетическим данным)
REGION_PARAMS = {
    "Республика Алтай":       dict(popbase=220,  tfrbase=2.45, migbase=-1.5, lebase=71.2),
    "Республика Бурятия":     dict(popbase=985,  tfrbase=2.10, migbase=-5.0, lebase=70.8),
    "Республика Тыва":        dict(popbase=335,  tfrbase=3.20, migbase=-3.0, lebase=66.5),
    "Республика Хакасия":     dict(popbase=537,  tfrbase=1.90, migbase=-2.0, lebase=70.5),
    "Алтайский край":         dict(popbase=2310, tfrbase=1.75, migbase=-8.0, lebase=71.8),
    "Забайкальский край":     dict(popbase=1060, tfrbase=2.05, migbase=-10.0, lebase=70.2),
    "Красноярский край":      dict(popbase=2855, tfrbase=1.85, migbase=-1.0, lebase=72.1),
    "Иркутская область":      dict(popbase=2380, tfrbase=1.95, migbase=-6.0, lebase=70.9),
    "Новосибирская область":  dict(popbase=2790, tfrbase=1.78, migbase= 2.0, lebase=73.0),
}

COLUMNS = [
    "year",
    "Численность населения всего",
    "СКР (всего)",
    "СКР первых детей",
    "СКР вторых детей",
    "СКР третьих и последующих детей",
    "Естественный прирост (на 1000)",
    "Миграционное сальдо",
    "Коэффициент миграционного прироста (на 10000)",
    "Ожидаемая продолжительность жизни",
    "Уровень безработицы (%)",
]
print("✓ REGION_PARAMS и базовые COLUMNS определены")

✓ REGION_PARAMS и базовые COLUMNS определены


In [8]:
def generate_synthetic_data(region_name: str, cfg: dict) -> pd.DataFrame:
    """
    Генерация синтетического демографического профиля 1991–histend
    с базовыми параметрами для региона.
    """
    histend = cfg["histend"]
    years = np.arange(1991, histend + 1)
    n = len(years)
    t = np.linspace(0, 1, n)

    params = REGION_PARAMS.get(region_name, dict(popbase=800, tfrbase=1.9, migbase=-3.0, lebase=71.0))

    rng = np.random.default_rng(SEED + abs(hash(region_name)) % 10_000)

    # population (тыс)
    pop_base = params["popbase"]
    pop = pop_base * (1 - 0.08 * t + 0.03 * t**2) + rng.normal(0, pop_base * 0.005, n)
    pop = np.maximum(pop, 50)

    # TFR
    tfr_base = params["tfrbase"]
    tfr_trend = np.where(
        years < 2000,
        -0.6 * (2000 - years) / 9,
        np.where(
            years < 2006,
            0.0,
            0.05 * (years - 2006) / 5,
        ),
    )
    tfr = tfr_base + tfr_trend + rng.normal(0, 0.05, n)
    tfr = np.clip(tfr, 0.9, 4.5)

    tfr1 = tfr * rng.uniform(0.38, 0.42, n)
    tfr2 = tfr * rng.uniform(0.30, 0.35, n)
    tfr3 = tfr - tfr1 - tfr2
    tfr3 = np.maximum(tfr3, 0)

    # natural increase per 1000
    natinc = (tfr - 2.1) * 5 + rng.normal(0, 0.5, n)
    natinc = np.clip(natinc, -12, 15)

    # migration per 10k
    mig_base = params["migbase"]
    mig = mig_base + 2.0 * t + rng.normal(0, 1.5, n)

    # life expectancy
    le_base = params["lebase"]
    le = le_base - 2.0 * (1 - t) + rng.normal(0, 0.3, n)
    le = np.clip(le, 60, 80)

    # unemployment
    unemp = 8.0 - 3.0 * t + rng.normal(0, 0.8, n)
    unemp = np.clip(unemp, 2.5, 18.0)

    df = pd.DataFrame(
        {
            "year": years,
            "Численность населения всего": np.round(pop * 1_000, 1),
            "СКР (всего)": np.round(tfr, 3),
            "СКР первых детей": np.round(tfr1, 3),
            "СКР вторых детей": np.round(tfr2, 3),
            "СКР третьих и последующих детей": np.round(tfr3, 3),
            "Естественный прирост (на 1000)": np.round(natinc, 2),
            "Миграционное сальдо": np.round(mig * pop, 2),
            "Коэффициент миграционного прироста (на 10000)": np.round(mig, 2),
            "Ожидаемая продолжительность жизни": np.round(le, 2),
            "Уровень безработицы (%)": np.round(unemp, 2),
        }
    )

    return df


print("✓ generate_synthetic_data готов")

✓ generate_synthetic_data готов


In [9]:
def load_region_data(region_name: str, cfg: dict) -> pd.DataFrame:
    """
    Загружает реальные данные региона из Excel/CSV.
    Синтетика запрещена.
    Поддерживает файлы, где первый столбец не year, а, например, 'Регион', 'Год', ...
    """
    datadir = Path(cfg["datadir"])
    sheet = cfg["sheet"]
    histend = cfg["histend"]
    trainend = cfg["trainend"]

    datadir.mkdir(parents=True, exist_ok=True)

    stem = region_name.replace(" ", "_").replace("-", "_")
    xlsx_path = datadir / f"{stem}.xlsx"
    csv_path = datadir / f"{stem}.csv"

    if xlsx_path.exists():
        try:
            df = pd.read_excel(xlsx_path, sheet_name=sheet)
            print(region_name, "→ Excel", xlsx_path)
        except Exception:
            df = pd.read_excel(xlsx_path)
            print(region_name, "→ Excel", xlsx_path, "(лист по умолчанию)")
    elif csv_path.exists():
        df = pd.read_csv(csv_path)
        print(region_name, "→ CSV", csv_path)
    else:
        raise FileNotFoundError(
            f"Для региона '{region_name}' не найден файл {xlsx_path.name} или {csv_path.name}"
        )

    df.columns = [str(c).strip() for c in df.columns]

    year_candidates = ["year", "Year", "Год", "год"]
    year_col = None
    for c in df.columns:
        if c in year_candidates:
            year_col = c
            break

    if year_col is None:
        raise ValueError(
            f"{region_name}: не найдена колонка года. Колонки файла: {list(df.columns)}"
        )

    if year_col != "year":
        df = df.rename(columns={year_col: "year"})

    rename_map = {
        "Численность постоянного населения на 1 января": "Численность населения всего",
        "Суммарный коэффициент рождаемости (всего)": "СКР (всего)",
        "Суммарный коэффициент рождаемости первых детей": "СКР первых детей",
        "Суммарный коэффициент рождаемости вторых детей": "СКР вторых детей",
        "Суммарный коэффициент рождаемости третьих и последующих детей": "СКР третьих и последующих детей",
        "Ожидаемая продолжительность жизни при рождении": "Ожидаемая продолжительность жизни",
        "Уровень безработицы (по методологии МОТ)": "Уровень безработицы (%)",
        "Общая площадь жилых помещений на 1 жителя, кв. м": "Общая площадь жилья на 1 жителя (кв. м)",
        "Валовой коэффициент охвата дошкольным образованием, %": "Валовой коэффициент охвата дошкольным образованием (%)",
        "Численность инвалидов, тыс. человек": "Число инвалидов (тыс.)",
        "Численность врачей, чел.": "Численность врачей",
        "Число больничных коек, ед.": "Число больничных коек",
        "Обеспеченность больничными койками (на 10 тыс. населения)": "Обеспеченность койками (на 10 тыс.)",
        "Число абортов, всего": "Число абортов (всего)",
    }
    df = df.rename(columns=rename_map)

    drop_cols = [c for c in ["Регион", "регион", "Тип", "type"] if c in df.columns]
    if drop_cols:
        df = df.drop(columns=drop_cols)

    df["year"] = pd.to_numeric(df["year"], errors="coerce")
    df = df.dropna(subset=["year"]).copy()
    df["year"] = df["year"].astype(int)

    df = df.sort_values("year").drop_duplicates(subset=["year"], keep="last").reset_index(drop=True)
    df = df[df["year"] <= histend].copy()

    for c in df.columns:
        if c != "year":
            df[c] = pd.to_numeric(df[c], errors="coerce")

    numeric_cols = [c for c in df.columns if c != "year"]
    keep_numeric = [c for c in numeric_cols if not df[c].isna().all()]
    df = df[["year"] + keep_numeric].copy()

    min_required = [
        "Численность населения всего",
        "СКР (всего)",
        "Миграционное сальдо",
        "Ожидаемая продолжительность жизни",
    ]
    missing_required = [c for c in min_required if c not in df.columns]
    if missing_required:
        raise ValueError(
            f"{region_name}: в файле не хватает обязательных колонок: {missing_required}"
        )

    train_mask = df["year"] <= trainend
    if train_mask.sum() < 2:
        raise ValueError(
            f"{region_name}: слишком мало наблюдений до trainend={trainend} для масштабирования"
        )

    scaler = StandardScaler()
    scaler.fit(df.loc[train_mask, keep_numeric].fillna(df[keep_numeric].median(numeric_only=True)))

    df_scaled = df.copy()
    fill_values = df_scaled[keep_numeric].median(numeric_only=True)
    df_scaled[keep_numeric] = df_scaled[keep_numeric].fillna(fill_values)
    df_scaled[keep_numeric] = scaler.transform(df_scaled[keep_numeric])

    df_scaled.attrs["scaler"] = scaler
    df_scaled.attrs["numcols"] = keep_numeric
    df_scaled.attrs["region"] = region_name

    return df_scaled


print("✓ load_region_data обновлена под реальные Excel-файлы")

✓ load_region_data обновлена под реальные Excel-файлы


In [10]:
# Загрузка данных всех 10 регионов
print("Загрузка данных по регионам СФО...")
region_data = {}
for r in CFG["regions"]:
    region_data[r] = load_region_data(r, CFG)
print("✓ Загружено регионов:", len(region_data))

Загрузка данных по регионам СФО...
Республика Алтай → Excel data/sfo/Республика_Алтай.xlsx
Республика Бурятия → Excel data/sfo/Республика_Бурятия.xlsx
Республика Тыва → Excel data/sfo/Республика_Тыва.xlsx
Республика Хакасия → Excel data/sfo/Республика_Хакасия.xlsx
Алтайский край → Excel data/sfo/Алтайский_край.xlsx
Забайкальский край → Excel data/sfo/Забайкальский_край.xlsx
Красноярский край → Excel data/sfo/Красноярский_край.xlsx
Иркутская область → Excel data/sfo/Иркутская_область.xlsx
Новосибирская область → Excel data/sfo/Новосибирская_область.xlsx
✓ Загружено регионов: 9


In [11]:
# EDA: сводная таблица по ключевым метрикам
eda_rows = []
key_metrics = [
    "Численность населения всего",
    "СКР (всего)",
    "Естественный прирост (на 1000)",
    "Коэффициент миграционного прироста (на 10000)",
    "Ожидаемая продолжительность жизни",
]

for region, df in region_data.items():
    row = {"region": region}
    for m in key_metrics:
        if m in df.columns:
            vals = df[m]
            row[f"{m} (min)"] = round(vals.min(), 3)
            row[f"{m} (mean)"] = round(vals.mean(), 3)
            row[f"{m} (max)"] = round(vals.max(), 3)
    eda_rows.append(row)

eda_table = pd.DataFrame(eda_rows).set_index("region")
print("EDA: min, mean, max по ключевым метрикам\n")
print(eda_table.to_string())

EDA: min, mean, max по ключевым метрикам

                       Численность населения всего (min)  Численность населения всего (mean)  Численность населения всего (max)  СКР (всего) (min)  СКР (всего) (mean)  СКР (всего) (max)  Ожидаемая продолжительность жизни (min)  Ожидаемая продолжительность жизни (mean)  Ожидаемая продолжительность жизни (max)
region                                                                                                                                                                                                                                                                                                               
Республика Алтай                                  -2.257                               0.952                              2.908             -1.260               0.211              2.275                                   -1.638                                     0.934                                    3.887
Республика Бурятия          

## Section 3 — Генерация данных до 2050 года

Экстраполяция исторических данных выполняется с помощью
простых регрессионных моделей с последующим сглаживанием и отсечками.

In [12]:
def smooth_clip(arr: np.ndarray, low: float, high: float, window: int = 7) -> np.ndarray:
    """
    Сглаживание рядом Савицкого–Голея и обрезка значений в [low, high].
    """
    if len(arr) < window:
        w = window if window % 2 == 1 else window + 1
        if len(arr) <= 2:
            return np.clip(arr, low, high)
    else:
        w = window if window % 2 == 1 else window + 1

    arr_sm = savgol_filter(arr, w, polyorder=2)
    return np.clip(arr_sm, low, high)


def fit_forecast_series(series: np.ndarray, n_future: int, nonnegative: bool = False) -> np.ndarray:
    """
    Линейная регрессия по историческим данным + экстраполяция на n_future шагов.
    """
    n = len(series)
    X = np.arange(n).reshape(-1, 1)
    model = LinearRegression()
    model.fit(X, series)

    X_fut = np.arange(n, n + n_future).reshape(-1, 1)
    pred = model.predict(X_fut)
    if nonnegative:
        pred = np.maximum(pred, 0)
    return pred

In [13]:
def extend_to_2050(df_region: pd.DataFrame, cfg: dict, region_name: str) -> pd.DataFrame:
    """
    Расширяет DataFrame региона до forecastend, используя регрессию + сглаживание.
    """
    histend = cfg["histend"]
    foreend = cfg["forecastend"]

    hist = df_region.copy()
    numcols = [c for c in hist.columns if c != "year"]

    future_years = np.arange(histend + 1, foreend + 1)
    n_future = len(future_years)

    ext_rows = {}

    for col in numcols:
        series = hist[col].values.astype(float)
        pred = fit_forecast_series(series, n_future, nonnegative=False)
        full = np.concatenate([series, pred])
        full = smooth_clip(full, full.min() - 0.5, full.max() + 0.5)
        ext_rows[col] = full

    all_years = np.concatenate([hist["year"].values, future_years])
    df_ext = pd.DataFrame({"year": all_years})
    for col in numcols:
        df_ext[col] = ext_rows[col]

    df_ext.attrs["scaler"] = hist.attrs.get("scaler")
    df_ext.attrs["numcols"] = numcols
    df_ext.attrs["region"] = region_name

    return df_ext


print("✓ extend_to_2050 готов")

region_extended = {
    r: extend_to_2050(region_data[r], CFG, r)
    for r in CFG["regions"]
}

print("✓ Region extended до 2050, пример хвоста:\n")
print(region_extended[CFG["regions"][0]].tail(5))

✓ extend_to_2050 готов
✓ Region extended до 2050, пример хвоста:

    year   Прибыло    Выбыло  Миграционное сальдо  \
56  2046  4.099679  5.178748            -1.994303   
57  2047  4.180966  5.283660            -2.039336   
58  2048  4.262253  5.388572            -2.084368   
59  2049  4.343540  5.493485            -2.129401   
60  2050  4.424827  5.598397            -2.174434   

    Численность населения всего  Ожидаемая продолжительность жизни  \
56                     6.350918                           5.141921   
57                     6.491150                           5.251216   
58                     6.631382                           5.360512   
59                     6.771614                           5.469807   
60                     6.911846                           5.579103   

    СКР (всего)  СКР вторых детей  СКР третьих и последующих детей  \
56     1.353898         -0.029098                        -0.011341   
57     1.383578         -0.029772                     

In [14]:
# График: Численность населения и СКР по всем регионам до 2050
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

colors = plt.cm.tab10(np.linspace(0, 1, len(CFG["regions"])))
ax1, ax2 = axes

for i, region in enumerate(CFG["regions"]):
    dfe = region_extended[region]
    ax1.plot(
        dfe["year"],
        dfe["Численность населения всего"],
        color=colors[i],
        label=region,
        linewidth=1.5,
    )
    ax2.plot(
        dfe["year"],
        dfe["СКР (всего)"],
        color=colors[i],
        label=region,
        linewidth=1.5,
    )

ax1.axvline(CFG["histend"], color="gray", linestyle="--", alpha=0.7, label="histend")
ax2.axvline(CFG["histend"], color="gray", linestyle="--", alpha=0.7)

ax1.set_title("Численность населения до 2050 г. (масштабированные значения)", fontsize=13)
ax1.set_ylabel("Население (scaled)")
ax1.set_xlabel("Год")
ax1.legend(fontsize=6.5, ncol=2, loc="lower left")
ax1.grid(alpha=0.3)

ax2.set_title("СКР до 2050 г. (масштабированные значения)", fontsize=13)
ax2.set_ylabel("СКР (scaled)")
ax2.set_xlabel("Год")
ax2.legend(fontsize=6.5, ncol=2, loc="lower right")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTDIR / "sec3_region_trends.png", bbox_inches="tight")
plt.close()
print("✓ Сохранён график:", PLOTDIR / "sec3_region_trends.png")

✓ Сохранён график: sfo_marl_output/plots/sec3_region_trends.png


## Section 4 — Постановка MARL-задачи

Определяются группы признаков, спецификации агентов, размеры
пространств состояний и действий для многоагентной среды.

In [15]:
# ── Section 4 · Feature groups & Agent specs (REAL XLSX COLUMNS) ─────────────

FEATURE_GROUPS = {
    "population_core": [
        "Численность населения всего",
        "Доля городского населения в общей численности населения на 1 января",
        "Уровень безработицы (%)",
        "Уровень бедности, %",
        "Доля населения с доходами ниже ПМ, %",
    ],

    "fertility": [
        "СКР (всего)",
        "СКР первых детей",
        "СКР вторых детей",
        "СКР третьих и последующих детей",
        "Число абортов (всего)",
    ],

    "migration": [
        "Прибыло",
        "Выбыло",
        "Миграционное сальдо",
    ],

    "household_economy": [
        "Денежные расходы домашних хозяйств",
        "Натуральный доход домашних хозяйств",
        "Располагаемые ресурсы домашних хозяйств",
        "Сумма сделанных сбережений домашних хозяйств",
        "Реальная начисленная заработная плата, % к предыдущему году",
        "Структура денежных доходов: Доходы от предпринимательской деятельности",
        "Структура денежных доходов: Доходы от собственности",
        "Структура денежных доходов: Другие доходы",
        "Структура денежных доходов: Оплата труда",
        "Структура денежных доходов: Социальные трансферты",
        "Структура расходов: Оплата услуг",
        "Структура расходов: Покупка алкогольных напитков",
        "Структура расходов: Покупка непродовольственных товаров",
        "Структура расходов: Покупка продуктов питания",
        "Структура расходов: Стоимость услуг, оказанных работодателем бесплатно или по льготным ценам",
    ],

    "housing": [
        "Введено в действие жилой площади, тыс. кв. м",
        "Общая площадь жилья на 1 жителя (кв. м)",
        "Число семей, нуждающихся в жилом помещении, тыс.",
        "Сведения о задолженности по кредитам: Жилищные кредиты",
        "Сведения о задолженности по кредитам: Ипотечные жилищные кредиты",
        "Сведения о задолженности по кредитам: По кредитам всего",
        "Сведения о задолженности по кредитам: в том числе: просроченная задолженность по ипотечным кредитам",
        "Индекс цен на первичном рынке жилья, %",
    ],

    "macro_costs": [
        "ВРП на душу населения, руб.",
        "Индекс физического объема ВРП на душу населения, %",
        "Индекс физического объема оборота розничной торговли, %",
        "Индекс физического объема платных услуг населению, %",
        "Индекс изменения стоимости фиксированного набора, %",
        "Индекс потребительских цен (декабрь к декабрю), %",
        "Стоимость фиксированного набора потребительских товаров и услуг, ₽",
        "Доля домохозяйств с широкополосным доступом к Интернету, %",
    ],

    "education_childcare": [
        "Валовой коэффициент охвата дошкольным образованием (%)",
        "Обеспеченность местами в ДОУ (на 1000 детей)",
    ],

    "healthcare_infra": [
        "Ожидаемая продолжительность жизни",
        "Мощность амбулаторно-поликлинических организаций, посещений в смену",
        "Обеспеченность койками (на 10 тыс.)",
        "Число инвалидов (тыс.)",
        "Численность врачей",
        "Число больничных коек",
        "Число посещений врачей, всего",
    ],

    "disease_burden": [
        "Число зарегистрированных заболеваний: COVID-19",
        "Число зарегистрированных заболеваний: Болезни глаза и придаточного аппарата",
        "Число зарегистрированных заболеваний: Болезни кожи и подкожной клетчатки",
        "Число зарегистрированных заболеваний: Болезни костно-мышечной системы и соединительной ткани",
        "Число зарегистрированных заболеваний: Болезни крови и кроветворных органов",
        "Число зарегистрированных заболеваний: Болезни мочеполовой системы",
        "Число зарегистрированных заболеваний: Болезни нервной системы",
        "Число зарегистрированных заболеваний: Болезни органов дыхания",
        "Число зарегистрированных заболеваний: Болезни органов пищеварения",
        "Число зарегистрированных заболеваний: Болезни системы кровообращения",
        "Число зарегистрированных заболеваний: Болезни уха и сосцевидного отростка",
        "Число зарегистрированных заболеваний: Болезни эндокринной системы, расстройства питания, нарушения обмена веществ и иммунитета",
        "Число зарегистрированных заболеваний: Всего заболеваний",
        "Число зарегистрированных заболеваний: Некоторые инфекционные и паразитарные болезни",
        "Число зарегистрированных заболеваний: Новообразования",
        "Число зарегистрированных заболеваний: Осложнения беременности, родов и послеродового периода",
        "Число зарегистрированных заболеваний: Психические расстройства и расстройства поведения",
        "Число зарегистрированных заболеваний: Симптомы, признаки и отклонения от нормы, выявленные при клинических и лабораторных исследованиях, не классифицированные в других рубриках",
        "Число зарегистрированных заболеваний: Состояния, возникающие в перинатальном периоде",
    ],
}

STATE_COLS = []
for cols in FEATURE_GROUPS.values():
    for c in cols:
        if c not in STATE_COLS:
            STATE_COLS.append(c)

STATE_DIM = len(STATE_COLS)
print("STATE_DIM:", STATE_DIM)

AGENT_SPECS = [
    {
        "id": 0,
        "name": "RegionalGov",
        "type": "institutional",
        "action_dim": 5,
        "action_names": [
            "family_support_budget",
            "housing_program_intensity",
            "social_benefit_access",
            "employment_support",
            "regional_info_policy",
        ],
        "obs_groups": [
            "population_core",
            "fertility",
            "migration",
            "housing",
            "macro_costs",
        ],
    },
    {
        "id": 1,
        "name": "HealthcareSystem",
        "type": "institutional",
        "action_dim": 4,
        "action_names": [
            "abortion_prevention",
            "reproductive_care_support",
            "doctor_staffing_support",
            "primary_care_capacity",
        ],
        "obs_groups": [
            "fertility",
            "healthcare_infra",
            "disease_burden",
        ],
    },
    {
        "id": 2,
        "name": "EducationInfra",
        "type": "institutional",
        "action_dim": 3,
        "action_names": [
            "preschool_capacity_expansion",
            "childcare_access_support",
            "digital_access_support",
        ],
        "obs_groups": [
            "education_childcare",
            "population_core",
            "macro_costs",
        ],
    },
    {
        "id": 3,
        "name": "MigrationPolicy",
        "type": "institutional",
        "action_dim": 3,
        "action_names": [
            "inflow_attraction",
            "outflow_retention",
            "settlement_support",
        ],
        "obs_groups": [
            "migration",
            "population_core",
            "macro_costs",
            "housing",
        ],
    },
    {
        "id": 4,
        "name": "Women",
        "type": "individual",
        "action_dim": 4,
        "action_names": [
            "birth_intention_1st",
            "birth_intention_2nd",
            "birth_intention_3plus",
            "abortion_avoidance",
        ],
        "obs_groups": [
            "fertility",
            "household_economy",
            "housing",
            "education_childcare",
            "healthcare_infra",
        ],
    },
    {
        "id": 5,
        "name": "Households",
        "type": "individual",
        "action_dim": 4,
        "action_names": [
            "housing_uptake",
            "consumption_reallocation",
            "saving_behavior",
            "credit_uptake",
        ],
        "obs_groups": [
            "household_economy",
            "housing",
            "macro_costs",
        ],
    },
    {
        "id": 6,
        "name": "Families",
        "type": "individual",
        "action_dim": 3,
        "action_names": [
            "family_expansion_preference",
            "childcare_usage",
            "stability_preference",
        ],
        "obs_groups": [
            "fertility",
            "household_economy",
            "housing",
            "education_childcare",
        ],
    },
    {
        "id": 7,
        "name": "Migrants",
        "type": "individual",
        "action_dim": 3,
        "action_names": [
            "arrival_propensity",
            "departure_avoidance",
            "labour_participation",
        ],
        "obs_groups": [
            "migration",
            "macro_costs",
            "population_core",
        ],
    },
]


def _obs_indices(groups):
    idx = []
    for grp in groups:
        for col in FEATURE_GROUPS.get(grp, []):
            if col in STATE_COLS:
                i = STATE_COLS.index(col)
                if i not in idx:
                    idx.append(i)
    return sorted(idx)


for spec in AGENT_SPECS:
    spec["obs_indices"] = _obs_indices(spec["obs_groups"])
    spec["obs_dim"] = len(spec["obs_indices"])
    print(
        f"Agent {spec['name']:20s}  "
        f"obs_dim={spec['obs_dim']:3d}  action_dim={spec['action_dim']}"
    )

N_AGENTS = len(AGENT_SPECS)
ACTION_DIMS = [s["action_dim"] for s in AGENT_SPECS]
ACTION_DIM_T = sum(ACTION_DIMS)
OBSDIMS = [s["obs_dim"] for s in AGENT_SPECS]
TOTALOBSDIM = sum(OBSDIMS)

print(f"[Env] N_agents={N_AGENTS}, ACTION_DIM_T={ACTION_DIM_T}, STATE_DIM={STATE_DIM}")

STATE_DIM: 72
Agent RegionalGov           obs_dim= 29  action_dim=5
Agent HealthcareSystem      obs_dim= 31  action_dim=4
Agent EducationInfra        obs_dim= 15  action_dim=3
Agent MigrationPolicy       obs_dim= 24  action_dim=3
Agent Women                 obs_dim= 37  action_dim=4
Agent Households            obs_dim= 31  action_dim=4
Agent Families              obs_dim= 30  action_dim=3
Agent Migrants              obs_dim= 16  action_dim=3
[Env] N_agents=8, ACTION_DIM_T=29, STATE_DIM=72


In [16]:
# Модель перехода среды — TransitionNet (ансамбль MLP)
class TransitionNet(nn.Module):
    """
    Простая MLP-модель перехода f(s_t, a_t) -> s_{t+1}.
    """

    def __init__(self, statedim: int, actiondim: int, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(statedim + actiondim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, statedim),
        )

    def forward(self, state, action):
        x = torch.cat([state, action], dim=-1)
        return self.net(x)


def build_transition_dataset(scaled_data: pd.DataFrame, actiondim: int, rng=None):
    if rng is None:
        rng = np.random.default_rng(SEED)

    numcols = [c for c in scaled_data.columns if c != "year"]
    vals = scaled_data[numcols].values.astype(np.float32)

    state_indices = [numcols.index(c) for c in STATE_COLS if c in numcols]

    states = vals[:-1, state_indices]
    next_states = vals[1:, state_indices]
    n = len(states)

    actions = rng.uniform(-1, 1, size=(n, actiondim)).astype(np.float32)

    return states, actions, next_states


def train_world_model(data: pd.DataFrame, cfg: dict, n_ensemble: int = 5):
    states_np, actions_np, next_states_np = build_transition_dataset(data, ACTION_DIM_T)

    models = [
        TransitionNet(STATE_DIM, ACTION_DIM_T, hidden=128).to(DEVICE)
        for _ in range(n_ensemble)
    ]
    optimizers = [optim.Adam(m.parameters(), lr=1e-3) for m in models]

    n = len(states_np)
    if n < 2:
        return models

    # Держим базовые тензоры на CPU, переносим на DEVICE внутри train-цикла
    states_cpu = torch.as_tensor(states_np, dtype=torch.float32, device="cpu")
    actions_cpu = torch.as_tensor(actions_np, dtype=torch.float32, device="cpu")
    next_states_cpu = torch.as_tensor(next_states_np, dtype=torch.float32, device="cpu")

    dataset = torch.utils.data.TensorDataset(states_cpu, actions_cpu, next_states_cpu)
    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=min(64, n),
        shuffle=True,
        pin_memory=False,  # КРИТИЧЕСКО: выключаем pin_memory, чтобы не пинить CUDA-тензоры
    )

    for model, opt in zip(models, optimizers):
        model.train()
        for _ in range(50):
            for sb_cpu, ab_cpu, nsb_cpu in loader:
                sb = sb_cpu.to(DEVICE, non_blocking=False)
                ab = ab_cpu.to(DEVICE, non_blocking=False)
                nsb = nsb_cpu.to(DEVICE, non_blocking=False)

                pred = model(sb, ab)
                loss = F.mse_loss(pred, nsb)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
        model.eval()

    return models


print("✓ TransitionNet, build_transition_dataset, train_world_model готовы")

✓ TransitionNet, build_transition_dataset, train_world_model готовы


In [17]:
def compute_reward(
    s_prev: np.ndarray,
    s_next: np.ndarray,
    joint_action: np.ndarray,
    agent_id: int,
    cfg: dict,
) -> float:
    """
    Композитная награда для агента agent_id.
    Использует относительные изменения демографических показателей и штраф за стоимость действий.
    """
    w = cfg["rewardw"]
    idx = {c: i for i, c in enumerate(STATE_COLS)}

    def d(col: str, scale: float = 1.0) -> float:
        if col not in idx:
            return 0.0
        i = idx[col]
        return float((s_next[i] - s_prev[i]) * scale)

    r = 0.0
    # глобальные метрики
    r += w["popgrowth"] * d("Численность населения всего", scale=1e-5)
    r += w["tfr"] * d("СКР (всего)", scale=1.0)
    r += w["netmigration"] * d("Миграционное сальдо", scale=1e-4)
    r += w["natincrease"] * d("Естественный прирост (на 1000)", scale=1.0)

    # инфраструктура
    r += w["housing"] * d("Общая площадь жилья на 1 жителя (кв. м)", scale=0.1)
    r += w["healthcare"] * d(
        "Ожидаемая продолжительность жизни", scale=0.1
    )

    # штраф за интенсивность действий
    cost = float(np.sum(joint_action**2))
    r += w["costpenalty"] * cost

    return float(r)

In [18]:
class RegionDemoEnv:
    """
    Среда демографического прогнозирования для региона СФО.
    Использует ансамбль TransitionNet как world-model.
    """

    def __init__(self, world_ensemble, init_state: np.ndarray, cfg: dict, region_name: str):
        self.ensemble = world_ensemble
        self.init_state = init_state.astype(np.float32).copy()
        self.cfg = cfg
        self.region_name = region_name

        self.state = self.init_state.copy()
        self.stepcount = 0

    def reset(self) -> np.ndarray:
        noise = np.random.normal(0, self.cfg["shockstd"], self.init_state.shape).astype(np.float32)
        self.state = self.init_state + noise
        self.stepcount = 0
        return self.state.copy()

    def get_obs(self, agent_id: int) -> np.ndarray:
        spec = AGENT_SPECS[agent_id]
        idx = spec["obs_indices"]
        return self.state[idx].copy()

    def step(self, joint_action: np.ndarray):
        st = torch.as_tensor(self.state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        at = torch.as_tensor(joint_action, dtype=torch.float32, device=DEVICE).unsqueeze(0)

        with torch.no_grad():
            preds = [m(st, at) for m in self.ensemble]
            next_state_t = torch.stack(preds, dim=0).mean(dim=0)

        next_state = next_state_t.squeeze(0).detach().cpu().numpy()
        next_state = next_state + np.random.normal(0, self.cfg["shockstd"], next_state.shape).astype(np.float32)

        rewards = [
            compute_reward(self.state, next_state, joint_action, i, self.cfg)
            for i in range(self.cfg["nagents"])
        ]

        self.state = next_state.astype(np.float32)
        self.stepcount += 1
        done = self.stepcount >= self.cfg["maxsteps"]

        info = {}
        return self.state.copy(), rewards, done, info


print("✓ RegionDemoEnv готов")

✓ RegionDemoEnv готов


## Section 5 — Архитектура MATD3

Реализация компонентов Multi-Agent Twin Delayed Deep Deterministic Policy Gradient:
- буфер воспроизведения;
- сети акторов и двойного критика;
- класс агента MATD3 и процедура обучения.

In [19]:
class ReplayBuffer:
    """
    Буфер воспроизведения опыта для MATD3.
    Хранит (obs, actions, rewards, next_obs, dones) для всех агентов.
    """

    def __init__(self, capacity: int, nagents: int, obsdims: list, actiondims: list):
        self.capacity = capacity
        self.nagents = nagents
        self.obsdims = obsdims
        self.actiondims = actiondims

        self.ptr = 0
        self.size = 0

        self.obs = [np.zeros((capacity, d), dtype=np.float32) for d in obsdims]
        self.actions = [np.zeros((capacity, d), dtype=np.float32) for d in actiondims]
        self.rewards = np.zeros((capacity, nagents), dtype=np.float32)
        self.nextobs = [np.zeros((capacity, d), dtype=np.float32) for d in obsdims]
        self.dones = np.zeros((capacity, 1), dtype=np.float32)

    def add(self, obs_list, action_list, reward_list, nextobs_list, done: bool):
        i = self.ptr
        for ag in range(self.nagents):
            self.obs[ag][i] = obs_list[ag]
            self.actions[ag][i] = action_list[ag]
            self.nextobs[ag][i] = nextobs_list[ag]
        self.rewards[i] = np.asarray(reward_list, dtype=np.float32)
        self.dones[i, 0] = float(done)

        self.ptr = (self.ptr + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batchsize: int):
        idx = np.random.choice(self.size, batchsize, replace=False)

        obs_batch = [torch.as_tensor(self.obs[ag][idx], dtype=torch.float32, device=DEVICE) for ag in range(self.nagents)]
        actions_batch = [torch.as_tensor(self.actions[ag][idx], dtype=torch.float32, device=DEVICE) for ag in range(self.nagents)]
        rewards_batch = torch.as_tensor(self.rewards[idx], dtype=torch.float32, device=DEVICE)
        nextobs_batch = [torch.as_tensor(self.nextobs[ag][idx], dtype=torch.float32, device=DEVICE) for ag in range(self.nagents)]
        dones_batch = torch.as_tensor(self.dones[idx], dtype=torch.float32, device=DEVICE)

        return obs_batch, actions_batch, rewards_batch, nextobs_batch, dones_batch

    def __len__(self):
        return self.size


print("✓ ReplayBuffer готов")

✓ ReplayBuffer готов


In [20]:
class Actor(nn.Module):
    """
    Сеть актора для MATD3.
    Отображает наблюдение агента в непрерывное действие ∈ [-1, 1]^d.
    """

    def __init__(self, obsdim: int, actiondim: int, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obsdim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, actiondim),
            nn.Tanh(),
        )

    def forward(self, obs):
        return self.net(obs)

In [21]:
class DoubleCritic(nn.Module):
    """
    Twin Critics для MATD3.
    Q1 и Q2 оценивают (obs_concat, act_concat); используется min-trick.
    """

    def __init__(self, totalobsdim: int, totalactdim: int, hidden: int = 256):
        super().__init__()
        indim = totalobsdim + totalactdim

        self.q1 = nn.Sequential(
            nn.Linear(indim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

        self.q2 = nn.Sequential(
            nn.Linear(indim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, obscat, actcat):
        x = torch.cat([obscat, actcat], dim=-1)
        return self.q1(x), self.q2(x)

    def q1only(self, obscat, actcat):
        x = torch.cat([obscat, actcat], dim=-1)
        return self.q1(x)


print("✓ Actor и DoubleCritic готовы")

✓ Actor и DoubleCritic готовы


In [22]:
class MATD3Agent:
    """
    Агент MATD3 (Multi-Agent Twin Delayed Deep Deterministic Policy Gradient).
    """

    def __init__(
        self,
        agent_id: int,
        obsdim: int,
        actiondim: int,
        totalobsdim: int,
        totalactdim: int,
        cfg: dict,
    ):
        self.id = agent_id
        self.obsdim = obsdim
        self.actiondim = actiondim

        self.gamma = cfg["gamma"]
        self.tau = cfg["tau"]
        self.policydelay = cfg["policydelay"]
        self.targetnoise = cfg["targetnoise"]
        self.noiseclip = cfg["noiseclip"]

        h = cfg["hiddendim"]

        self.actor = Actor(obsdim, actiondim, h).to(DEVICE)
        self.actortarget = copy.deepcopy(self.actor).to(DEVICE)
        self.actoropt = optim.Adam(self.actor.parameters(), lr=cfg["actorlr"])

        self.critic = DoubleCritic(totalobsdim, totalactdim, h).to(DEVICE)
        self.critictarget = copy.deepcopy(self.critic).to(DEVICE)
        self.criticopt = optim.Adam(self.critic.parameters(), lr=cfg["criticlr"])

        self.updatecount = 0

    def select_action(self, obs: np.ndarray, noise_scale: float = 0.0) -> np.ndarray:
        obs_t = torch.as_tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            act = self.actor(obs_t).squeeze(0).detach().cpu().numpy()
        if noise_scale > 0.0:
            act = act + np.random.normal(0, noise_scale, size=act.shape).astype(np.float32)
        return np.clip(act, -1.0, 1.0).astype(np.float32)

    def update(self, batch, all_agents: list, agent_idx: int):
        obs_b, act_b, rew_b, nextobs_b, done_b = batch
        self.updatecount += 1

        with torch.no_grad():
            next_acts = []
            for ag in all_agents:
                na = ag.actortarget(nextobs_b[ag.id])
                noise = torch.clamp(
                    torch.randn_like(na) * self.targetnoise,
                    -self.noiseclip,
                    self.noiseclip,
                )
                na = torch.clamp(na + noise, -1.0, 1.0)
                next_acts.append(na)

            next_obs_cat = torch.cat(nextobs_b, dim=-1)
            next_act_cat = torch.cat(next_acts, dim=-1)

            q1_next, q2_next = self.critictarget(next_obs_cat, next_act_cat)
            q_next = torch.min(q1_next, q2_next)

            reward_i = rew_b[:, agent_idx].unsqueeze(1)
            y = reward_i + self.gamma * (1.0 - done_b) * q_next

        obs_cat = torch.cat(obs_b, dim=-1)
        act_cat = torch.cat(act_b, dim=-1)

        q1, q2 = self.critic(obs_cat, act_cat)
        critic_loss = F.mse_loss(q1, y) + F.mse_loss(q2, y)

        self.criticopt.zero_grad(set_to_none=True)
        critic_loss.backward()
        self.criticopt.step()

        actor_loss_val = None

        if self.updatecount % self.policydelay == 0:
            acts_for_update = []
            for k, ag in enumerate(all_agents):
                if k == agent_idx:
                    acts_for_update.append(ag.actor(obs_b[ag.id]))
                else:
                    with torch.no_grad():
                        acts_for_update.append(ag.actor(obs_b[ag.id]))

            act_cat_new = torch.cat(acts_for_update, dim=-1)
            actor_loss = -self.critic.q1only(obs_cat, act_cat_new).mean()

            self.actoropt.zero_grad(set_to_none=True)
            actor_loss.backward()
            self.actoropt.step()
            actor_loss_val = float(actor_loss.item())

            self.soft_update(self.actor, self.actortarget)
            self.soft_update(self.critic, self.critictarget)

        return {
            "critic_loss": float(critic_loss.item()),
            "actor_loss": actor_loss_val,
        }

    def soft_update(self, src, tgt):
        tau = self.tau
        for p, tp in zip(src.parameters(), tgt.parameters()):
            tp.data.copy_(tau * p.data + (1.0 - tau) * tp.data)


print("✓ MATD3Agent готов")

✓ MATD3Agent готов


In [23]:
def run_matd3_training(
    cfg: dict,
    world_ensemble,
    init_state: np.ndarray,
    region_name: str,
):
    """
    Запуск обучения MATD3 для одного региона.
    Возвращает список агентов и словарь логов.
    """
    env = RegionDemoEnv(world_ensemble, init_state, cfg, region_name)

    agents = [
        MATD3Agent(
            agent_id=i,
            obsdim=OBSDIMS[i],
            actiondim=ACTION_DIMS[i],
            totalobsdim=TOTALOBSDIM,
            totalactdim=ACTION_DIM_T,
            cfg=cfg,
        )
        for i in range(cfg["nagents"])
    ]

    buffer = ReplayBuffer(
        cfg["buffersize"], cfg["nagents"], OBSDIMS, ACTION_DIMS
    )

    logs = {
        "ep_return": [],
        "critic_losses": [],
        "actor_losses": [],
    }

    noise_scale = 0.3

    for ep in range(cfg["nepisodes"]):
        env.reset()
        ep_ret = 0.0
        ep_critic_losses = []
        ep_actor_losses = []

        for _ in range(cfg["maxsteps"]):
            obs_list = [env.get_obs(i) for i in range(cfg["nagents"])]

            act_list = [
                agents[i].select_action(obs_list[i], noise_scale)
                for i in range(cfg["nagents"])
            ]

            joint_act = np.concatenate(act_list, axis=0).astype(np.float32)

            _, rewards, done, _ = env.step(joint_act)
            next_obs_list = [env.get_obs(i) for i in range(cfg["nagents"])]

            buffer.add(obs_list, act_list, rewards, next_obs_list, done)

            ep_ret += float(sum(rewards))

            if len(buffer) >= max(cfg["warmupsteps"], cfg["batchsize"]):
                batch = buffer.sample(cfg["batchsize"])
                for i, ag in enumerate(agents):
                    info = ag.update(batch, agents, i)
                    ep_critic_losses.append(info["critic_loss"])
                    if info["actor_loss"] is not None:
                        ep_actor_losses.append(info["actor_loss"])

            if done:
                break

        noise_scale = max(0.05, noise_scale * 0.997)

        logs["ep_return"].append(ep_ret)
        logs["critic_losses"].append(float(np.mean(ep_critic_losses)) if ep_critic_losses else 0.0)
        logs["actor_losses"].append(float(np.mean(ep_actor_losses)) if ep_actor_losses else 0.0)

        if (ep + 1) % 100 == 0 or ep == 0:
            print(
                f"[{region_name}] ep {ep+1}/{cfg['nepisodes']}  "
                f"return={ep_ret:.2f}  noise={noise_scale:.3f}"
            )

    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return agents, logs


print("✓ run_matd3_training готов")

✓ run_matd3_training готов


## Section 6 — Обучение MATD3 по всем регионам СФО

Для каждого из 10 регионов:
- обучение ансамбля TransitionNet;
- обучение многоагентной политики MATD3;
- сохранение весов акторов/критиков и логов обучения.

In [25]:
print("=" * 100)
print("ПРОВЕРКА СООТВЕТСТВИЯ STATE_COLS: файл на диске vs region_data в памяти")
print("=" * 100)

datadir = Path(CFG["datadir"])
file_map = CFG.get("file_map", {})

for region in CFG["regions"]:
    print(f"\n[REGION] {region}")

    if region in file_map:
        xlsx_path = datadir / file_map[region]
        csv_path = datadir / Path(file_map[region]).with_suffix(".csv").name
    else:
        stem = region.replace(" ", "_").replace("-", "_")
        xlsx_path = datadir / f"{stem}.xlsx"
        csv_path = datadir / f"{stem}.csv"

    chosen_path = None
    if xlsx_path.exists():
        chosen_path = xlsx_path
    elif csv_path.exists():
        chosen_path = csv_path

    print("  expected file:", chosen_path if chosen_path is not None else "NOT FOUND")

    # 1) Проверка файла на диске
    if chosen_path is not None:
        try:
            if chosen_path.suffix.lower() == ".xlsx":
                df_file = pd.read_excel(chosen_path, sheet_name=CFG.get("sheet", 0))
            else:
                df_file = pd.read_csv(chosen_path)

            df_file.columns = [str(c).strip() for c in df_file.columns]

            file_cols = list(df_file.columns)
            if "Год" in file_cols and "year" not in file_cols:
                file_cols_norm = ["year" if c == "Год" else c for c in file_cols]
            else:
                file_cols_norm = file_cols

            file_numcols = [c for c in file_cols_norm if c not in ["year", "Регион", "регион", "Тип", "type"]]

            rename_map_reverse_check = {
                "Численность постоянного населения на 1 января": "Численность населения всего",
                "Суммарный коэффициент рождаемости (всего)": "СКР (всего)",
                "Суммарный коэффициент рождаемости первых детей": "СКР первых детей",
                "Суммарный коэффициент рождаемости вторых детей": "СКР вторых детей",
                "Суммарный коэффициент рождаемости третьих и последующих детей": "СКР третьих и последующих детей",
                "Ожидаемая продолжительность жизни при рождении": "Ожидаемая продолжительность жизни",
                "Уровень безработицы (по методологии МОТ)": "Уровень безработицы (%)",
                "Общая площадь жилых помещений на 1 жителя, кв. м": "Общая площадь жилья на 1 жителя (кв. м)",
                "Валовой коэффициент охвата дошкольным образованием, %": "Валовой коэффициент охвата дошкольным образованием (%)",
                "Численность инвалидов, тыс. человек": "Число инвалидов (тыс.)",
                "Численность врачей, чел.": "Численность врачей",
                "Число больничных коек, ед.": "Число больничных коек",
                "Обеспеченность больничными койками (на 10 тыс. населения)": "Обеспеченность койками (на 10 тыс.)",
                "Число абортов, всего": "Число абортов (всего)",
            }

            file_numcols_norm = [rename_map_reverse_check.get(c, c) for c in file_numcols]
            file_missing = [c for c in STATE_COLS if c not in file_numcols_norm]

            print(f"  file columns (usable): {len(file_numcols_norm)}")
            print(f"  file missing from STATE_COLS: {len(file_missing)}")
            if file_missing:
                print("    file missing sample:", file_missing[:10])

        except Exception as e:
            print("  file check error:", repr(e))
    else:
        print("  file check skipped: no source file found")

    # 2) Проверка region_data в памяти
    if region in region_data:
        df_hist = region_data[region]
        mem_numcols = [c for c in df_hist.columns if c != "year"]
        mem_missing = [c for c in STATE_COLS if c not in mem_numcols]

        print(f"  memory columns (usable): {len(mem_numcols)}")
        print(f"  memory missing from STATE_COLS: {len(mem_missing)}")
        if mem_missing:
            print("    memory missing sample:", mem_missing[:10])
    else:
        print("  memory check skipped: region not in region_data")

    # 3) Короткий вывод
    if chosen_path is not None and region in region_data:
        if len(file_missing) == 0 and len(mem_missing) > 0:
            print("  >>> ВНИМАНИЕ: файл на диске уже хороший, но region_data не перезагружен.")
        elif len(file_missing) == 0 and len(mem_missing) == 0:
            print("  >>> OK: и файл, и region_data согласованы.")
        elif len(file_missing) > 0:
            print("  >>> ПРОБЛЕМА В ФАЙЛЕ: даже файл не содержит полный STATE_COLS.")
        else:
            print("  >>> НЕОЖИДАННОЕ РАСХОЖДЕНИЕ, проверь load_region_data.")

ПРОВЕРКА СООТВЕТСТВИЯ STATE_COLS: файл на диске vs region_data в памяти

[REGION] Республика Алтай
  expected file: data/sfo/Республика_Алтай.xlsx
  file columns (usable): 72
  file missing from STATE_COLS: 0
  memory columns (usable): 72
  memory missing from STATE_COLS: 0
  >>> OK: и файл, и region_data согласованы.

[REGION] Республика Бурятия
  expected file: data/sfo/Республика_Бурятия.xlsx
  file columns (usable): 72
  file missing from STATE_COLS: 0
  memory columns (usable): 72
  memory missing from STATE_COLS: 0
  >>> OK: и файл, и region_data согласованы.

[REGION] Республика Тыва
  expected file: data/sfo/Республика_Тыва.xlsx
  file columns (usable): 72
  file missing from STATE_COLS: 0
  memory columns (usable): 72
  memory missing from STATE_COLS: 0
  >>> OK: и файл, и region_data согласованы.

[REGION] Республика Хакасия
  expected file: data/sfo/Республика_Хакасия.xlsx
  file columns (usable): 72
  file missing from STATE_COLS: 0
  memory columns (usable): 72
  memory mi

In [26]:
# Цикл обучения по всем регионам с одним progress bar и ETA — ускоренная версия
region_agents = {}
region_logs = {}
region_world_models = {}

with region_experiment_bar(CFG["regions"], desc="MATD3 regions") as (pbar, exp_start):
    total_regions = len(CFG["regions"])

    for idx, region in enumerate(CFG["regions"], start=1):
        region_start = time.time()

        df_hist = region_data[region]
        numcols_hist = [c for c in df_hist.columns if c != "year"]
        state_indices = [numcols_hist.index(c) for c in STATE_COLS if c in numcols_hist]
        last_row = df_hist[numcols_hist].values[-1]
        init_state = last_row[state_indices].astype(np.float32)

        world_ensemble = train_world_model(df_hist, CFG)
        agents, logs = run_matd3_training(CFG, world_ensemble, init_state, region)

        region_agents[region] = agents
        region_logs[region] = logs
        region_world_models[region] = world_ensemble

        fname = region.replace(" ", "_").replace("-", "_")
        for i, ag in enumerate(agents):
            torch.save(ag.actor.state_dict(), MODELDIR / f"{fname}_agent{i}_actor.pt")
            torch.save(ag.critic.state_dict(), MODELDIR / f"{fname}_agent{i}_critic.pt")

        region_elapsed = time.time() - region_start
        update_region_bar(
            pbar,
            exp_start,
            done_count=idx,
            total_count=total_regions,
            region_name=region,
            region_elapsed=region_elapsed,
        )
        pbar.update(1)

        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

print("✓ Обучение MATD3 по всем регионам завершено")

MATD3 regions:   0%|          | 0/9 [00:00<?, ?it/s]

[Республика Алтай] ep 1/600  return=-26.18  noise=0.299
[Республика Алтай] ep 100/600  return=-28.78  noise=0.222
[Республика Алтай] ep 200/600  return=-26.38  noise=0.164
[Республика Алтай] ep 300/600  return=-54.19  noise=0.122
[Республика Алтай] ep 400/600  return=-218.43  noise=0.090
[Республика Алтай] ep 500/600  return=-262.65  noise=0.067
[Республика Алтай] ep 600/600  return=-276.46  noise=0.050
[Республика Бурятия] ep 1/600  return=-26.10  noise=0.299
[Республика Бурятия] ep 100/600  return=-26.27  noise=0.222
[Республика Бурятия] ep 200/600  return=-19.98  noise=0.164
[Республика Бурятия] ep 300/600  return=-37.31  noise=0.122
[Республика Бурятия] ep 400/600  return=-147.69  noise=0.090
[Республика Бурятия] ep 500/600  return=-227.56  noise=0.067
[Республика Бурятия] ep 600/600  return=-240.51  noise=0.050
[Республика Тыва] ep 1/600  return=-28.54  noise=0.299
[Республика Тыва] ep 100/600  return=-26.24  noise=0.222
[Республика Тыва] ep 200/600  return=-30.73  noise=0.164
[Ре

In [27]:
# График кривых обучения по всем регионам
fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.tab10(np.linspace(0, 1, len(CFG["regions"])))

for i, region in enumerate(CFG["regions"]):
    returns = region_logs[region]["ep_return"]
    window = 20
    if len(returns) > window:
        smoothed = pd.Series(returns).rolling(window).mean().values
    else:
        smoothed = returns
    ax.plot(smoothed, color=colors[i], label=region, linewidth=1.5)

ax.set_title("MATD3: скользящее среднее вознаграждения (20 эп.)", fontsize=13)
ax.set_xlabel("Эпизод")
ax.set_ylabel("Вознаграждение (сумма по агентам)")
ax.legend(fontsize=7, ncol=2, loc="lower right")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTDIR / "sec6_learning_curves.png", bbox_inches="tight")
plt.close()
print("✓ Сохранён график:", PLOTDIR / "sec6_learning_curves.png")

✓ Сохранён график: sfo_marl_output/plots/sec6_learning_curves.png


## Section 7 — Оценка: траектории и baseline

Сравнение сценариев:
- **baseline** — нулевые действия (joint_action = 0),
- **MATD3** — действия обученных агентов.

In [28]:
def rollout(
    agents,
    world_ensemble,
    init_state: np.ndarray,
    cfg: dict,
    region_name: str,
    stochastic: bool = False,
    zero_action: bool = False,
) -> pd.DataFrame:
    """
    Прогон траектории на горизонте cfg['horizon'].
    Если zero_action=True, используется baseline (все действия = 0).
    """
    env = RegionDemoEnv(world_ensemble, init_state, cfg, region_name)
    state = env.reset()

    rows = []
    rows.append(dict(zip(STATE_COLS, state.tolist()), step=0))

    for step in range(1, cfg["horizon"] + 1):
        obs_list = [env.get_obs(i) for i in range(cfg["nagents"])]

        if zero_action:
            act_list = [np.zeros(ACTION_DIMS[i], dtype=np.float32) for i in range(cfg["nagents"])]
        else:
            noise = 0.05 if stochastic else 0.0
            act_list = [
                agents[i].select_action(obs_list[i], noise_scale=noise)
                for i in range(cfg["nagents"])
            ]

        joint_act = np.concatenate(act_list, axis=0)
        next_state, _, done, _ = env.step(joint_act)

        rows.append(dict(zip(STATE_COLS, next_state.tolist()), step=step))
        if done:
            break

    return pd.DataFrame(rows)


print("✓ rollout готов")

✓ rollout готов


In [29]:
# Вычисление траекторий baseline и MATD3 для всех регионов — без повторного обучения world model
baseline_trajs = {}
matd3_trajs = {}

for region in CFG["regions"]:
    df_hist = region_data[region]
    numcols_hist = [c for c in df_hist.columns if c != "year"]
    state_indices = [numcols_hist.index(c) for c in STATE_COLS if c in numcols_hist]
    init_state = df_hist[numcols_hist].values[-1, state_indices].astype(np.float32)

    world_ensemble = region_world_models[region]

    baseline_trajs[region] = rollout(
        region_agents[region],
        world_ensemble,
        init_state,
        CFG,
        region,
        zero_action=True,
    )

    matd3_trajs[region] = rollout(
        region_agents[region],
        world_ensemble,
        init_state,
        CFG,
        region,
        zero_action=False,
    )

print("✓ Траектории baseline и MATD3 посчитаны")

✓ Траектории baseline и MATD3 посчитаны


In [30]:
# Расчёт эффекта: % разница MATD3 vs baseline в конце горизонта
effect_size = {}
key_metrics_traj = [
    "Численность населения всего",
    "СКР (всего)",
    "Естественный прирост (на 1000)",
    "Коэффициент миграционного прироста (на 10000)",
]

for region in CFG["regions"]:
    b_last = baseline_trajs[region].iloc[-1]
    m_last = matd3_trajs[region].iloc[-1]
    effect_size[region] = {}
    for col in key_metrics_traj:
        if col in b_last.index:
            bval = b_last[col]
            mval = m_last[col]
            delta = mval - bval
            if abs(bval) < 1e-9:
                pct = np.nan
            else:
                pct = 100.0 * delta / bval
            effect_size[region][col] = round(pct, 2)

effect_df = pd.DataFrame(effect_size).T
print("MATD3 vs baseline (% к baseline в конце горизонта):\n")
print(effect_df.to_string())

MATD3 vs baseline (% к baseline в конце горизонта):

                       Численность населения всего  СКР (всего)
Республика Алтай                             55.75      -134.53
Республика Бурятия                           60.76       -29.01
Республика Тыва                               2.92       -18.38
Республика Хакасия                         1485.75       -92.70
Алтайский край                              -83.14       -68.84
Забайкальский край                         -158.31        12.48
Красноярский край                           -45.78         2.87
Иркутская область                            60.28       -78.28
Новосибирская область                      -161.92       -26.99


## Section 8 — Анализ действий агентов

Сбор и анализ предпочтений действий каждого агента:
- средние уровни действий по годам и регионам;
- тепловая карта значимости действий.

In [31]:
def collect_action_log(
    agents,
    world_ensemble,
    init_state: np.ndarray,
    cfg: dict,
    region_name: str,
    n_rollouts: int = 10,
) -> pd.DataFrame:
    env = RegionDemoEnv(world_ensemble, init_state, cfg, region_name)
    records = []

    for _ in range(n_rollouts):
        env.reset()
        for step in range(cfg["maxsteps"]):
            obs_list = [env.get_obs(i) for i in range(cfg["nagents"])]

            actions = []
            for i, ag in enumerate(agents):
                action = ag.select_action(obs_list[i], noise_scale=0.0)
                spec = AGENT_SPECS[i]
                for j, val in enumerate(action):
                    records.append(
                        {
                            "region": region_name,
                            "agent_name": spec["name"],
                            "action_index": j,
                            "action_name": f"{spec['name']}__{spec['action_names'][j]}",
                            "value": float(val),
                        }
                    )
                actions.append(action)

            joint_act = np.concatenate(actions, axis=0).astype(np.float32)
            _, _, done, _ = env.step(joint_act)
            if done:
                break

    return pd.DataFrame(records)


print("✓ collect_action_log готов")

# Сбор логов действий для всех регионов — без повторного обучения world model
action_logs = {}

for region in CFG["regions"]:
    df_hist = region_data[region]
    numcols_hist = [c for c in df_hist.columns if c != "year"]
    state_indices = [numcols_hist.index(c) for c in STATE_COLS if c in numcols_hist]
    init_state = df_hist[numcols_hist].values[-1, state_indices].astype(np.float32)

    world_ensemble = region_world_models[region]

    action_logs[region] = collect_action_log(
        region_agents[region],
        world_ensemble,
        init_state,
        CFG,
        region,
        n_rollouts=10,
    )

all_action_log = pd.concat(action_logs.values(), ignore_index=True)

action_pref = (
    all_action_log.groupby(["region", "agent_name", "action_name"])["value"]
    .agg(mean="mean", std="std", meanabs=lambda x: x.abs().mean())
    .reset_index()
)
action_pref["std"] = action_pref["std"].fillna(0.0)

print("Топ-10 действий по |среднему уровню|:\n")
print(
    action_pref.sort_values("meanabs", ascending=False)
    .head(10)[["region", "agent_name", "action_name", "meanabs"]]
    .to_string(index=False)
)

✓ collect_action_log готов
Топ-10 действий по |среднему уровню|:

               region      agent_name                                  action_name  meanabs
   Забайкальский край      Households                   Households__housing_uptake      1.0
     Республика Алтай MigrationPolicy           MigrationPolicy__inflow_attraction      1.0
     Республика Алтай  EducationInfra     EducationInfra__childcare_access_support      1.0
    Красноярский край        Families                    Families__childcare_usage      1.0
   Республика Хакасия  EducationInfra       EducationInfra__digital_access_support      1.0
     Республика Алтай MigrationPolicy           MigrationPolicy__outflow_retention      1.0
     Республика Алтай      Households                    Households__credit_uptake      1.0
     Республика Алтай  EducationInfra EducationInfra__preschool_capacity_expansion      1.0
Новосибирская область      Households                   Households__housing_uptake      1.0
    Иркутская 

## Section 9 — Визуализация

Четыре основных графика:
1. Прогноз численности населения (baseline vs MATD3) по регионам.
2. Сравнение СКР в 2050 г. (baseline vs MATD3).
3. Тепловая карта интенсивности действий агентов.
4. Кривые обучения MATD3 по регионам.

In [32]:
# Plot 1: Динамика численности населения по годам (baseline vs MATD3)
n_reg = len(CFG["regions"])
ncols = 4
nrows = (n_reg + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3.5), sharex=True)
axes_flat = axes.flatten()

for i, region in enumerate(CFG["regions"]):
    ax = axes_flat[i]
    bdf = baseline_trajs[region]
    mdf = matd3_trajs[region]

    years_forecast = np.arange(CFG["histend"] + 1, CFG["histend"] + 1 + len(bdf))

    ax.plot(
        years_forecast,
        bdf["Численность населения всего"].values,
        color="steelblue",
        label="baseline",
        linewidth=1.5,
    )
    ax.plot(
        years_forecast,
        mdf["Численность населения всего"].values,
        color="darkorange",
        label="MATD3",
        linewidth=1.5,
        linestyle="--",
    )
    ax.set_title(region, fontsize=8)
    ax.set_ylabel("Население (scaled)", fontsize=7)
    ax.tick_params(labelsize=7)
    ax.grid(alpha=0.3)
    if i == 0:
        ax.legend(fontsize=7)

for j in range(n_reg, len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle("Численность населения до 2050 г.: baseline vs MATD3", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(PLOTDIR / "plot1_population_forecast.png", bbox_inches="tight")
plt.close()
print("✓ Plot 1 сохранён")

✓ Plot 1 сохранён


In [33]:
# Plot 2: 2050 — СКР (всего) MATD3 vs baseline
col_tfr = "СКР (всего)"

x = np.arange(len(CFG["regions"]))
width = 0.35

baseline_tfr = [
    baseline_trajs[r].iloc[-1][col_tfr] if col_tfr in baseline_trajs[r].columns else np.nan
    for r in CFG["regions"]
]
matd3_tfr = [
    matd3_trajs[r].iloc[-1][col_tfr] if col_tfr in matd3_trajs[r].columns else np.nan
    for r in CFG["regions"]
]

shortnames = [r.replace("Республика ", "Р. ").replace("край", "кр.") for r in CFG["regions"]]

fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - width / 2, baseline_tfr, width, label="baseline", color="steelblue", alpha=0.85)
ax.bar(x + width / 2, matd3_tfr, width, label="MATD3", color="darkorange", alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(shortnames, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("СКР (scaled)", fontsize=10)
ax.set_title("СКР (всего) в 2050 г.: MATD3 vs baseline", fontsize=12)
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTDIR / "plot2_tfr_comparison.png", bbox_inches="tight")
plt.close()
print("✓ Plot 2 сохранён")

✓ Plot 2 сохранён


In [34]:
# Plot 3: тепловая карта интенсивности действий агентов
pivot_heat = (
    action_pref.pivot_table(
        index="action_name",
        columns="region",
        values="meanabs",
        aggfunc="mean",
    )
    .fillna(0.0)
)

shortmap = {
    r: r.replace("Республика ", "Р. ").replace("край", "кр.") for r in CFG["regions"]
}
pivot_heat = pivot_heat.rename(columns=shortmap)

fig, ax = plt.subplots(
    figsize=(16, max(8, len(pivot_heat) * 0.35))
)
sns.heatmap(
    pivot_heat,
    ax=ax,
    cmap="YlOrRd",
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    cbar_kws={"label": "среднее |действие|"},
    annot_kws={"size": 6},
)

ax.set_title("Интенсивность действий агентов по регионам (|среднее значение|)", fontsize=12)
ax.set_xlabel("Регион", fontsize=10)
ax.set_ylabel("Действие", fontsize=10)
ax.tick_params(axis="x", labelrotation=35, labelsize=8)
ax.tick_params(axis="y", labelsize=7)

plt.tight_layout()
plt.savefig(PLOTDIR / "plot3_action_heatmap.png", bbox_inches="tight")
plt.close()
print("✓ Plot 3 сохранён")

✓ Plot 3 сохранён


In [35]:
# Plot 4: кривые обучения MATD3 по регионам СФО
fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.tab10(np.linspace(0, 1, len(CFG["regions"])))

for i, region in enumerate(CFG["regions"]):
    returns = region_logs[region]["ep_return"]
    window = 30
    if len(returns) > window:
        smoothed = pd.Series(returns).rolling(window, min_periods=1).mean().values
    else:
        smoothed = np.array(returns)
    shortname = region.replace("Республика ", "Р. ").replace("край", "кр.")
    ax.plot(smoothed, color=colors[i], label=shortname, linewidth=1.5)

ax.set_title("MATD3: кривые обучения по регионам СФО", fontsize=13)
ax.set_xlabel("Эпизод")
ax.set_ylabel("Скользящее среднее вознаграждения (30 эп.)")
ax.legend(fontsize=7, ncol=2, loc="lower right")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTDIR / "plot4_learning_curves.png", bbox_inches="tight")
plt.close()
print("✓ Plot 4 сохранён")
print("Папка графиков:", PLOTDIR)

✓ Plot 4 сохранён
Папка графиков: sfo_marl_output/plots


## Section 10 — Сводные таблицы

Итоговые таблицы:
- прогнозные значения по ключевым метрикам в 2030, 2040, 2050 годах;
- изменения (Δ, %) относительно baseline;
- приоритетные действия агентов (топ-3 по |среднему уровню|).

In [36]:
# Summary table: регион × метрика × год: baseline, matd3, delta%
horizon_years = {2030: 5, 2040: 15, 2050: 25}  # шаги от histend
histend = CFG["histend"]

summary_rows = []

for region in CFG["regions"]:
    bdf = baseline_trajs[region]
    mdf = matd3_trajs[region]
    for year, step in horizon_years.items():
        idx = min(step, len(bdf) - 1)
        brow = bdf.iloc[idx]
        mrow = mdf.iloc[idx]
        for col in key_metrics_traj:
            if col in brow.index:
                bval = brow[col]
                mval = mrow[col]
                delta = mval - bval
                if abs(bval) < 1e-9:
                    pct = np.nan
                else:
                    pct = 100.0 * delta / bval
                summary_rows.append(
                    {
                        "region": region,
                        "metric": col,
                        "year": year,
                        "Baseline": round(bval, 4),
                        "MATD3": round(mval, 4),
                        "Delta_%": round(pct, 2),
                    }
                )

summary_table = pd.DataFrame(summary_rows)
summary_csv = REPORTDIR / "summary_table.csv"
summary_table.to_csv(summary_csv, index=False, encoding="utf-8-sig")

print("✓ Summary таблица сохранена в", summary_csv)
print("Всего строк:", len(summary_table))
print(summary_table[summary_table["year"] == 2050].head(10).to_string(index=False))

✓ Summary таблица сохранена в sfo_marl_output/reportdata/summary_table.csv
Всего строк: 54
            region                      metric  year  Baseline   MATD3  Delta_%
  Республика Алтай Численность населения всего  2050    0.0720  0.1121    55.75
  Республика Алтай                 СКР (всего)  2050    0.0214 -0.0074  -134.53
Республика Бурятия Численность населения всего  2050    0.0485  0.0779    60.76
Республика Бурятия                 СКР (всего)  2050    0.0372  0.0264   -29.01
   Республика Тыва Численность населения всего  2050    0.2654  0.2732     2.92
   Республика Тыва                 СКР (всего)  2050    0.0573  0.0468   -18.38
Республика Хакасия Численность населения всего  2050    0.0019  0.0296  1485.75
Республика Хакасия                 СКР (всего)  2050    0.0407  0.0030   -92.70
    Алтайский край Численность населения всего  2050   -0.0159 -0.0027   -83.14
    Алтайский край                 СКР (всего)  2050    0.0786  0.0245   -68.84


In [ ]:
# Action priority table: топ-3 действия каждого агента по всем регионам
top3_rows = []

for region in CFG["regions"]:
    for spec in AGENT_SPECS:
        aname = spec["name"]
        subset = action_pref[
            (action_pref["region"] == region)
            & (action_pref["agent_name"] == aname)
        ]
        subset = subset.sort_values("meanabs", ascending=False).head(3)
        for _, row in subset.iterrows():
            top3_rows.append(
                {
                    "region": region,
                    "agent_name": aname,
                    "action_name": row["action_name"],
                    "meanabs": round(row["meanabs"], 4),
                }
            )

top3_table = pd.DataFrame(top3_rows)
top3_csv = REPORTDIR / "action_top3_table.csv"
top3_table.to_csv(top3_csv, index=False, encoding="utf-8-sig")

print("✓ Action priority таблица сохранена в", top3_csv)
print(top3_table.head(10).to_string(index=False))

In [37]:
# Итоговый отчёт
print("=" * 65)
print("           ИТОГИ ЭКСПЕРИМЕНТА SFO MARL (MATD3)")
print("=" * 65)
print("Горизонт:", CFG["histend"] + 1, "—", CFG["forecastend"])
print("Регионов СФО:", len(CFG["regions"]))
print("Агентов:", N_AGENTS)
print("STATE_DIM:", STATE_DIM, " ACTION_DIM_T:", ACTION_DIM_T)
print("Графики сохранены в:", PLOTDIR)
print("Таблицы сохранены в:", REPORTDIR)
print("=" * 65)

           ИТОГИ ЭКСПЕРИМЕНТА SFO MARL (MATD3)
Горизонт: 2026 — 2050
Регионов СФО: 9
Агентов: 8
STATE_DIM: 72  ACTION_DIM_T: 29
Графики сохранены в: sfo_marl_output/plots
Таблицы сохранены в: sfo_marl_output/reportdata


In [38]:
# ЛЁГКИЙ АРХИВ ДЛЯ ОТЧЁТА (без MODELDIR, только графики/таблицы/логи/ноутбук)
import os
import zipfile
from pathlib import Path
from datetime import datetime

run_tag = datetime.now().strftime("%Y%m%d_%H%M%S")
archive_name = f"sfo_marl_matd3_report_{run_tag}.zip"
archive_path = Path.cwd() / archive_name

paths_to_pack = [
    PLOTDIR,
    LOGDIR,
    REPORTDIR,
]

extra_files = []
for fname in Path.cwd().glob("*.ipynb"):
    extra_files.append(fname)

def add_path_to_zip(zf: zipfile.ZipFile, path: Path, arc_prefix: str = ""):
    path = path.resolve()
    if not path.exists():
        return

    if path.is_file():
        arcname = Path(arc_prefix) / path.name if arc_prefix else Path(path.name)
        zf.write(path, arcname.as_posix())
        return

    for file in path.rglob("*"):
        if file.is_file():
            try:
                rel = file.relative_to(path.parent)
            except Exception:
                rel = Path(file.name)
            arcname = Path(arc_prefix) / rel if arc_prefix else rel
            zf.write(file, arcname.as_posix())

with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    packed = set()

    for p in paths_to_pack:
        p = Path(p)
        if p.exists():
            add_path_to_zip(zf, p)
            packed.add(str(p.resolve()))

    for f in extra_files:
        f = Path(f)
        if f.exists() and str(f.resolve()) not in packed:
            zf.write(f, f.name)

print("✓ Архив для отчёта создан:", archive_path)
print(f"Размер: {archive_path.stat().st_size / (1024 * 1024):.2f} MB")

# Автоскачивание, если это Google Colab
try:
    from google.colab import files
    print("Запускаю автоматическое скачивание архива...")
    files.download(str(archive_path))
except Exception as e:
    print("Автоскачивание не запущено (скорее всего, это не Colab).")
    print("Причина:", repr(e))
    print("Скачай архив вручную по пути:", archive_path)

✓ Архив для отчёта создан: /content/sfo_marl_matd3_report_20260601_182337.zip
Размер: 0.98 MB
Запускаю автоматическое скачивание архива...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>